In [12]:
import json
import re
import sys
import html
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "collection_notebooks" else Path.cwd()
project_root_str = str(PROJECT_ROOT.resolve())

if project_root_str not in sys.path:
    sys.path.append(project_root_str)

from src.utils.hackernews_client import hackerNewsItem, searchHackerNewsStories

In [4]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "collection_notebooks" else Path.cwd()

RAW_DIR = PROJECT_ROOT / "data/raw/hackernews"
PROCESSED_DIR = PROJECT_ROOT / "data/processed/hackernews"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MAX_STORIES_PER_QUERY = 20
MAX_COMMENTS_PER_STORY = 30
MAX_COMMENT_DEPTH = 1

SEARCH_QUERIES = [
    "GitHub Copilot",
    "Claude Code",
    "OpenAI Codex",
    "AI coding assistant",
    "vibe coding",
    "AI replacing programmers",
    "AI code security privacy",
    "AI software developer productivity",
]

In [5]:
def unix_to_iso(unix_time):
    """
    Convert Hacker News Unix timestamp into readable ISO date.
    """
    if not unix_time:
        return None

    return datetime.fromtimestamp(unix_time, tz=timezone.utc).strftime("%Y-%m-%d")


def fetch_comment_tree(commentIds, maxComments=200, maxDepth=3, parentId=None, depth=1):
    """
    Fetch comments from Hacker News.

    This also collects replies to comments up to maxDepth.
    """

    comments = []

    if not commentIds or maxComments <= 0 or depth > maxDepth:
        return comments

    for commentId in commentIds:
        if len(comments) >= maxComments:
            break

        try:
            comment = hackerNewsItem(commentId)
        except Exception as e:
            print(f"Could not fetch comment {commentId}: {e}")
            continue

        if not comment:
            continue

        if comment.get("deleted") or comment.get("dead"):
            continue

        if comment.get("type") == "comment":
            comment_record = {
                "commentId": comment.get("id"),
                "parentId": parentId,
                "author": comment.get("by"),
                "text": comment.get("text") or "",
                "createdAt": unix_to_iso(comment.get("time")),
                "depth": depth,
                "childCommentCount": len(comment.get("kids", [])),
            }

            comments.append(comment_record)

            remaining = maxComments - len(comments)

            child_comments = fetch_comment_tree(
                comment.get("kids", []),
                maxComments=remaining,
                maxDepth=maxDepth,
                parentId=comment.get("id"),
                depth=depth + 1,
            )

            comments.extend(child_comments)

    return comments[:maxComments]


def fetchHackerNewsData(
    searchQuery,
    maxStories=50,
    maxCommentsPerStory=200,
    maxCommentDepth=3,
    outputFile="hackernewsDataDump.json"
):
    """
    Search Hacker News stories for a query, then collect each story and comments.
    """

    story_hits = searchHackerNewsStories(searchQuery, maxStories=maxStories)
    story_threads = []
    seen_story_ids = set()

    for searchRank, hit in enumerate(story_hits, start=1):
        story_id = int(hit.get("objectID"))

        if story_id in seen_story_ids:
            continue

        seen_story_ids.add(story_id)

        try:
            story = hackerNewsItem(story_id)
        except Exception as e:
            print(f"Could not fetch story {story_id}: {e}")
            continue

        if not story:
            continue

        if story.get("type") != "story":
            continue

        if story.get("deleted") or story.get("dead"):
            continue

        comments = fetch_comment_tree(
            story.get("kids", []),
            maxComments=maxCommentsPerStory,
            maxDepth=maxCommentDepth,
            parentId=story_id,
            depth=1,
        )

        story_threads.append({
            "sourceQuery": searchQuery,
            "searchRank": searchRank,
            "story": story,
            "comments": comments,
        })

    data = {
        "source": "hackernews",
        "searchQuery": searchQuery,
        "storyThreads": story_threads,
    }

    with open(outputFile, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(story_threads)} Hacker News stories to {outputFile}")

    return data

In [10]:
print("Hacker News collection config:")
print("Search queries:", SEARCH_QUERIES)
print("MAX_STORIES_PER_QUERY:", MAX_STORIES_PER_QUERY)
print("MAX_COMMENTS_PER_STORY:", MAX_COMMENTS_PER_STORY)
print("MAX_COMMENT_DEPTH:", MAX_COMMENT_DEPTH)

for index, searchQuery in enumerate(SEARCH_QUERIES, start=1):
    output_file = RAW_DIR / f"hackernews_ai_coding_{index}.json"

    print(f"\nCollecting '{searchQuery}' -> {output_file}")

    fetchHackerNewsData(
        searchQuery,
        maxStories=MAX_STORIES_PER_QUERY,
        maxCommentsPerStory=MAX_COMMENTS_PER_STORY,
        maxCommentDepth=MAX_COMMENT_DEPTH,
        outputFile=str(output_file),
    )

Hacker News collection config:
Search queries: ['GitHub Copilot', 'Claude Code', 'OpenAI Codex', 'AI coding assistant', 'vibe coding', 'AI replacing programmers', 'AI code security privacy', 'AI software developer productivity']
MAX_STORIES_PER_QUERY: 20
MAX_COMMENTS_PER_STORY: 30
MAX_COMMENT_DEPTH: 1

Saved 20 Hacker News stories to C:\Users\aryan\Music\Social Media Analytics\social-media-2\data\raw\hackernews\hackernews_ai_coding_1.json

Saved 20 Hacker News stories to C:\Users\aryan\Music\Social Media Analytics\social-media-2\data\raw\hackernews\hackernews_ai_coding_2.json

Saved 20 Hacker News stories to C:\Users\aryan\Music\Social Media Analytics\social-media-2\data\raw\hackernews\hackernews_ai_coding_3.json

Saved 20 Hacker News stories to C:\Users\aryan\Music\Social Media Analytics\social-media-2\data\raw\hackernews\hackernews_ai_coding_4.json

Saved 20 Hacker News stories to C:\Users\aryan\Music\Social Media Analytics\social-media-2\data\raw\hackernews\hackernews_ai_coding_5.js

In [6]:
raw_files = sorted(RAW_DIR.glob("hackernews_ai_coding_*.json"))

if not raw_files:
    raise FileNotFoundError("No raw Hacker News files found. Run the collection cell first.")

story_rows = []
comment_rows = []
text_rows = []

seen_story_ids = set()
seen_comment_ids = set()

for json_file in raw_files:
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    for thread in data.get("storyThreads", []):
        story = thread.get("story", {})
        story_id = story.get("id")
        story_key = f"hn_{story_id}"

        if story_id not in seen_story_ids:
            seen_story_ids.add(story_id)

            story_rows.append({
                "sourceQuery": thread.get("sourceQuery"),
                "searchRank": thread.get("searchRank"),
                "storyKey": story_key,
                "storyId": story_id,
                "title": story.get("title", ""),
                "text": story.get("text") or "",
                "author": story.get("by"),
                "score": story.get("score", 0),
                "createdAt": unix_to_iso(story.get("time")),
                "url": story.get("url"),
                "hnUrl": f"https://news.ycombinator.com/item?id={story_id}",
                "commentCount": story.get("descendants", 0),
                "collectedCommentCount": len(thread.get("comments", [])),
            })

            text_rows.append({
                "source": "hackernews",
                "sourceQuery": thread.get("sourceQuery"),
                "threadKey": story_key,
                "recordType": "story",
                "recordId": story_id,
                "author": story.get("by"),
                "createdAt": unix_to_iso(story.get("time")),
                "text": f"{story.get('title', '')}\n\n{story.get('text') or ''}",
                "url": f"https://news.ycombinator.com/item?id={story_id}",
            })

        for comment in thread.get("comments", []):
            comment_id = comment.get("commentId")

            if comment_id in seen_comment_ids:
                continue

            seen_comment_ids.add(comment_id)

            comment_rows.append({
                "sourceQuery": thread.get("sourceQuery"),
                "storyKey": story_key,
                "storyId": story_id,
                "storyTitle": story.get("title", ""),
                "commentId": comment_id,
                "parentId": comment.get("parentId"),
                "commentAuthor": comment.get("author"),
                "commentText": comment.get("text") or "",
                "commentCreatedAt": comment.get("createdAt"),
                "commentDepth": comment.get("depth"),
                "childCommentCount": comment.get("childCommentCount", 0),
                "hnUrl": f"https://news.ycombinator.com/item?id={comment_id}",
            })

            text_rows.append({
                "source": "hackernews",
                "sourceQuery": thread.get("sourceQuery"),
                "threadKey": story_key,
                "recordType": "comment",
                "recordId": comment_id,
                "author": comment.get("author"),
                "createdAt": comment.get("createdAt"),
                "text": comment.get("text") or "",
                "url": f"https://news.ycombinator.com/item?id={comment_id}",
            })

stories_df = pd.DataFrame(story_rows)
comments_df = pd.DataFrame(comment_rows)
text_df = pd.DataFrame(text_rows)

stories_df.to_csv(PROCESSED_DIR / "HNStoriesRawFlattened.csv", index=False)
comments_df.to_csv(PROCESSED_DIR / "HNCommentsRawFlattened.csv", index=False)
text_df.to_csv(PROCESSED_DIR / "HNDiscussionTextRawFlattened.csv", index=False)

print("Stories:", stories_df.shape)
print("Comments:", comments_df.shape)
print("Text records:", text_df.shape)

stories_df.head()

Stories: (152, 13)
Comments: (2207, 12)
Text records: (2359, 9)


,sourceQuery,searchRank,storyKey,storyId,title,text,author,score,createdAt,url,hnUrl,commentCount,collectedCommentCount
0,GitHub Copilot,1,hn_27676266,27676266,GitHub Copilot,,todsacerdoti,2905,2021-06-29T14:29:39+00:00,https://copilot.github.com/,https://news.ycombinator.com/item?id=27676266,1272,30
1,GitHub Copilot,2,hn_35261065,35261065,GitHub Copilot X – Sign up for technical preview,,todsacerdoti,1096,2023-03-22T13:59:07+00:00,https://github.blog/2023-03-22-github-copilot-...,https://news.ycombinator.com/item?id=35261065,751,30
2,GitHub Copilot,3,hn_27687450,27687450,GitHub Copilot as open source code laundering?,,agomez314,1028,2021-06-30T12:00:41+00:00,https://twitter.com/eevee/status/1410037309848...,https://news.ycombinator.com/item?id=27687450,459,30
3,GitHub Copilot,4,hn_33226515,33226515,"GitHub Copilot, with “public code” blocked, em...",,davidgerard,914,2022-10-16T19:33:52+00:00,https://twitter.com/docsparse/status/158146173...,https://news.ycombinator.com/item?id=33226515,775,30
4,GitHub Copilot,5,hn_35921375,35921375,GitHub Copilot Chat Leaked Prompt,,marvinvonhagen,910,2023-05-12T19:48:59+00:00,https://twitter.com/marvinvonhagen/status/1657...,https://news.ycombinator.com/item?id=35921375,609,30


In [13]:
print("Hacker News dataset summary")
print("---------------------------")

print("Number of stories:", len(stories_df))
print("Number of comments:", len(comments_df))
print("Number of text records:", len(text_df))

if len(stories_df):
    stories_df["createdAt"] = pd.to_datetime(stories_df["createdAt"], errors="coerce")
    print("Story date range:", stories_df["createdAt"].min().date(), "to", stories_df["createdAt"].max().date())

if len(comments_df):
    comments_df["commentCreatedAt"] = pd.to_datetime(comments_df["commentCreatedAt"], errors="coerce")
    print("Comment date range:", comments_df["commentCreatedAt"].min().date(), "to", comments_df["commentCreatedAt"].max().date())

Hacker News dataset summary
---------------------------
Number of stories: 152
Number of comments: 2207
Number of text records: 2353
Story date range: 2018-03-27 to 2026-05-14
Comment date range: 2018-03-27 to 2026-05-07


In [14]:
def clean_hackernews_text(text):
    """
    Clean Hacker News story/comment text.
    """

    text = str(text)

    # Decode HTML entities such as &#x2F;, &amp;, &quot;
    text = html.unescape(text)

    # Remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Keep letters, numbers, spaces and apostrophes
    text = re.sub(r"[^A-Za-z0-9\s']", " ", text)

    # Remove repeated whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


text_df = text_df[text_df["text"].notna()].copy()

text_df["textClean"] = text_df["text"].apply(clean_hackernews_text)
text_df["textLower"] = text_df["textClean"].str.lower()
text_df["textLength"] = text_df["textClean"].str.len()

text_df = text_df[text_df["textLength"] > 0].copy()

text_df = text_df.drop_duplicates(
    subset=["threadKey", "recordType", "recordId", "textClean"]
).copy()

text_df.to_csv(PROCESSED_DIR / "hackernews_text_cleaned.csv", index=False)

print("Clean text records:", text_df.shape)

text_df.head()

Clean text records: (2351, 12)


,source,sourceQuery,threadKey,recordType,recordId,author,createdAt,text,url,textClean,textLower,textLength
0,hackernews,GitHub Copilot,hn_27676266,story,27676266,todsacerdoti,2021-06-29 14:29:39+00:00,GitHub Copilot\n\n,https://news.ycombinator.com/item?id=27676266,GitHub Copilot,github copilot,14
1,hackernews,GitHub Copilot,hn_27676266,comment,27682218,dang,2021-06-29 21:53:17+00:00,"To see all the 1100+ comments, you&#x27;ll nee...",https://news.ycombinator.com/item?id=27682218,To see all the 1100 comments you'll need to cl...,to see all the 1100 comments you'll need to cl...,225
2,hackernews,GitHub Copilot,hn_27676266,comment,27676845,fzaninotto,2021-06-29 15:05:28+00:00,I&#x27;ve been using the alpha for the past 2 ...,https://news.ycombinator.com/item?id=27676845,I've been using the alpha for the past 2 weeks...,i've been using the alpha for the past 2 weeks...,599
3,hackernews,GitHub Copilot,hn_27676266,comment,27676939,natfriedman,2021-06-29 15:10:45+00:00,"Hi HN, we&#x27;ve been building GitHub Copilot...",https://news.ycombinator.com/item?id=27676939,Hi HN we've been building GitHub Copilot toget...,hi hn we've been building github copilot toget...,393
4,hackernews,GitHub Copilot,hn_27676266,comment,27678728,cbsks,2021-06-29 17:07:30+00:00,I don&#x27;t think we need to start looking fo...,https://news.ycombinator.com/item?id=27678728,I don't think we need to start looking for new...,i don't think we need to start looking for new...,681


In [16]:
relevance_keywords = [
    "copilot",
    "cursor",
    "claude",
    "codex",
    "ai",
    "agent",
    "coding",
    "programmer",
    "developer",
    "productivity",
    "trust",
    "hallucinat",
    "bug",
    "security",
    "privacy",
    "cost",
    "subscription",
    "ownership",
]

for keyword in relevance_keywords:
    count = text_df["textLower"].str.contains(re.escape(keyword), regex=True).sum()
    print(f"{keyword}: {count}")

copilot: 305
cursor: 57
claude: 413
codex: 113
ai: 1282
agent: 238
coding: 414
programmer: 70
developer: 167
productivity: 76
trust: 57
hallucinat: 25
bug: 118
security: 55
privacy: 39
cost: 99
subscription: 47
ownership: 3


In [17]:
stop_words = {
    "the", "and", "that", "for", "this", "you", "with", "but", "not",
    "are", "have", "can", "they", "was", "more", "will", "how", "all",
    "your", "there", "from", "what", "just", "would", "like", "use",
    "about", "into", "then", "than", "when", "which", "their", "them",
    "these", "those", "also", "been", "because", "could", "should",
    "were", "has", "had", "his", "her", "our", "out", "get", "got",
    "one", "two", "some", "any", "who", "why", "where", "way", "see",
    "make", "much", "many", "very", "really", "even", "still", "only",
    "does", "did", "doing", "done", "over", "under", "between", "after",
    "before", "while", "through", "using", "used", "being", "same",
    "well", "think", "people", "thing", "things"
}

word_counter = Counter()

for text in text_df["textLower"]:
    words = re.findall(r"[a-z][a-z0-9_]{2,}", str(text))

    useful_words = []

    for word in words:
        if word not in stop_words:
            useful_words.append(word)

    word_counter.update(useful_words)

common_words = word_counter.most_common(30)

common_terms_df = pd.DataFrame(common_words, columns=["term", "frequency"])
common_terms_df.to_csv(PROCESSED_DIR / "hackernews_common_terms.csv", index=False)

common_terms_df

,term,frequency
0,code,1903
1,claude,685
2,coding,556
3,copilot,476
4,time,467
5,don,454
6,work,412
7,good,341
8,now,337
9,vibe,322


In [18]:
user_story_rows = []

for _, row in comments_df.iterrows():
    comment_author = row.get("commentAuthor")
    story_key = row.get("storyKey")

    if pd.notna(comment_author) and pd.notna(story_key):
        user_story_rows.append({
            "source": f"user_{comment_author}",
            "target": story_key,
            "relationship": "commented_on_story",
            "weight": 1,
        })

user_story_edges = pd.DataFrame(user_story_rows)

if len(user_story_edges):
    user_story_edges = (
        user_story_edges
        .groupby(["source", "target", "relationship"], as_index=False)
        .agg(weight=("weight", "sum"))
    )

user_story_edges.to_csv(PROCESSED_DIR / "hackernews_user_story_edges.csv", index=False)

print("User-story edge table:", user_story_edges.shape)

user_story_edges.head()

User-story edge table: (2178, 4)


,source,target,relationship,weight
0,user_0-bad-sectors,hn_47762901,commented_on_story,1
1,user_0xWTF,hn_44966856,commented_on_story,1
2,user_0xbadcafebee,hn_45163362,commented_on_story,1
3,user_0xbadcafebee,hn_47167931,commented_on_story,1
4,user_0xcb0,hn_43163011,commented_on_story,1
